# ETL edges 

In [1]:
import requests 
import pandas as pd
import psycopg2
import os
from dotenv import load_dotenv
import json

api van https://geodata.antwerpen.be/arcgissql gebruiken om dataset edges van Antwerpen te importeren

In [2]:
api = "https://geodata.antwerpen.be/arcgissql/rest/services/P_Portal/portal_publiek4/MapServer/297/query?where=WEGNUMMER%20%3D%20'R1'%20AND%20WEGCAT%20%3D%20'hoofdweg'&outFields=WS_OIDN,B_WK_OIDN,E_WK_OIDN,STATUS,MORF,WEGCAT,LSTRNMID,LSTRNM,RSTRNMID,RSTRNM,BEHEER,METHODE,OPNDATUM,BEGINTIJD,BEGINORG,TGBEP,WS_UIDN,WS_GIDN,GBKA_ID,WEGNUMMER,WEGKLASSE,SNELHEID,RIJRICHTING_AUTO,CAT_MOBILITEITSPLAN,BOVENLOKAAL,LABEL,Shape_Length&outSR=4326&f=json"

In [3]:
response = requests.get(api)
print(response)

<Response [200]>


In [4]:
response_data = response.json()
#response_data['features'][0] #kijken hoe data er uit ziet

gegevens parsen in een df

In [5]:
df_edges_attributes = pd.DataFrame([response_data['features'][x]['attributes'] for x in range(len(response_data['features']))]) #parsen van edge data
df_edges_geometry = pd.DataFrame([response_data['features'][x]['geometry'] for x in range(len(response_data['features']))]) #parsen van geometry edge
df_edges_combined = pd.DataFrame(pd.concat([df_edges_attributes, df_edges_geometry], axis=1)) #df concat

In [6]:
mask = ['WS_OIDN','B_WK_OIDN','E_WK_OIDN', 'SNELHEID', 'RIJRICHTING_AUTO', 'Shape_Length', 'paths'] #gewenste kolommen
df_edges = df_edges_combined[mask]
df_edges.head(1) 

,WS_OIDN,B_WK_OIDN,E_WK_OIDN,SNELHEID,RIJRICHTING_AUTO,Shape_Length,paths
0,421566,2085123,1079886,100,enkel (mee),782.191221,"[[[4.421230557666013, 51.19099637960401], [4.4..."


Coördinaten staan in het verkeerde formaat. Onderstaande functie zal coördinaten recursief omdraaien 

In [7]:
def change_coord(coordinates):
    """
    Deze functie zal coordinaten wisselen van plaats [long, lat] -> [lat, long]
    """

    if isinstance(coordinates[0], list):
        return [change_coord(line) for line in coordinates]
    elif isinstance(coordinates[0], float):  
        return [coordinates[1], coordinates[0]]

In [8]:
df_edges.loc[:, 'paths'] = pd.Series([change_coord(linestring) for linestring in df_edges['paths']]) #kolom wisselen met paths in juist formaat

Verwijderen van edges die niet gevisualiseert mogen worden

In [9]:
removed_edges = [556092, 477787, 411870, 555120, 1177710, 1154275, 
                 1154268, 1154267, 1198422, 1198444, 421567, 536729, 
                 539943, 478156, 421594, 421595, 475072, 536365,
                 499035, 545294, 475084, 555690, 1192085, 1192086,
                 1154254, 1154280, 1154255, 1154257, 1154259, 1198423,
                 1198443, 1154258, 1154284, 1154285
                ]

df_edges = df_edges[[edge not in removed_edges for edge in df_edges['WS_OIDN']]] 

db gegevens uit .env bestand uitlezen 

In [10]:
load_dotenv()

host=os.getenv('DB_HOST')
dbname=os.getenv('POSTGRES_DB')
user=os.getenv('POSTGRES_USER')
password=os.getenv('POSTGRES_PASSWORD')
port=os.getenv('DB_PORT')

df_edges opladen naar database

In [11]:
conn = psycopg2.connect(host=host, dbname=dbname, user=user, password=password, port=port) 
cur = conn.cursor()

In [12]:
cur.execute("""CREATE TABLE IF NOT EXISTS edges(
    edge_id INT PRIMARY KEY,
    begin_node INT,
    end_node INT,
    speed SMALLINT,
    lane_direction TEXT,
    edge_length DOUBLE PRECISION,
    path JSONB
)
""")

conn.commit()

In [13]:
for i, row in df_edges.iterrows():
    values = (row.loc['WS_OIDN'], row.loc['B_WK_OIDN'], row.loc['E_WK_OIDN'], row.loc['SNELHEID'], row.loc['RIJRICHTING_AUTO'], row.loc['Shape_Length'], json.dumps(row.loc['paths']))
    cur.execute("""INSERT INTO edges(edge_id, begin_node, end_node, speed, lane_direction, edge_length, path) VALUES (%s, %s, %s, %s, %s, %s, %s)""", values)

conn.commit()

In [14]:
cur.close()
conn.close()